# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Lap Times Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

lap_times_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("race_name", StringType(), True),
    StructField("circuit_id", StringType(), True),
    StructField("lap", IntegerType(), True),
    StructField("driver_id", StringType(), False),
    StructField("position", IntegerType(), True),
    StructField("time", StringType(), True),
])

lap_times_input_path = f"{processed_folder_path}/lap_times/csv/lap_times.csv"

lap_times_final_df = spark.read \
    .option("header", True) \
    .schema(lap_times_schema) \
    .csv(lap_times_input_path)


# 3) Transform Lap Times Data:

The steps included:

- Create Surrogate Key.
- Add Data Source and File Date.
- Fill Null cells with "None".

In [0]:
from pyspark.sql.functions import lit

lap_times_with_audit_df = lap_times_final_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

lap_times_date_df = add_ingestion_date(lap_times_with_audit_df)
lap_times_fill_df = lap_times_date_df.fillna("None")

lap_times_final_df = add_surrogate_key(
    lap_times_fill_df,
    key_column_name="lap_times_sk",
    hash_columns=["season", "round", "race_name", "circuit_id", "lap", "driver_id", 
                  "position", "time"],
)

print("Final columns going into the write:", lap_times_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
lap_times_output_path = f"{processed_folder_path}/lap_times/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=lap_times_final_df,
    db_name="f1_processed",
    table_name="lap_times",
    output_path=lap_times_output_path,
    merge_key_columns=["season", "circuit_id"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(lap_times_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/lap_times/delta",
    presentation_directory=f"{presentation_folder_path}/fact_lap_times/delta",
    db_name="f1_presentation",
    table_name="fact_lap_times",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_lap_times/delta"))

# 5) Save backup Lap Times in CSV format:

In [0]:
import io
import csv

lap_times_backup_path = f"{presentation_folder_path}/fact_lap_times/csv/fact_lap_times.csv"

backup_rows = [row.asDict() for row in lap_times_final_df.collect()]
backup_fieldnames = lap_times_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(lap_times_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {lap_times_backup_path}")